## Poverty NN activation sweep (step-by-step)

This notebook reproduces what we did in a **sequential**, explain-as-we-go workflow:

- Build `poverty_lag1` from the panel data
- Time split (train `<=2020`, test `>=2021`)
- Baseline evaluation (`poverty_lag1`)
- PyTorch MLP training with early stopping
- Activation/architecture sweep and result table


## Step 1 — Load data + build the lag feature

We use `data/processed/panel/marz_year_panel_full.csv`, then compute:

- `poverty_lag1` = previous year’s `poverty_rate` within each `marz`.

This is the same baseline idea you used in the notebook for forecasting.


In [5]:
DATA_PATH = "../data/processed/panel/marz_year_panel_full.csv"

df = pd.read_csv(DATA_PATH)
df = df.sort_values(["marz", "year"]).copy()
df["poverty_lag1"] = df.groupby("marz")["poverty_rate"].shift(1)

print("rows, cols:", df.shape)
print("year min/max:", df["year"].min(), df["year"].max())
df[["marz", "year", "poverty_rate", "poverty_lag1"]].head(12)


rows, cols: (187, 25)
year min/max: 2008 2024


,marz,year,poverty_rate,poverty_lag1
1,Aragatsotn,2008,20.3,NaN
12,Aragatsotn,2009,25.4,20.3
23,Aragatsotn,2010,28.9,25.4
34,Aragatsotn,2011,20.7,28.9
45,Aragatsotn,2012,21.2,20.7
56,Aragatsotn,2013,22.7,21.2
67,Aragatsotn,2014,18.7,22.7
78,Aragatsotn,2015,16.1,18.7
89,Aragatsotn,2016,15.7,16.1
100,Aragatsotn,2017,17.6,15.7


## Step 2 — Define features/target and a time-based split

Features mirror what was in your earlier notebook (plus we *compute* `poverty_lag1` here).

- Train: years `<= 2020`
- Test: years `>= 2021`

Then we compute the baseline metrics using only `poverty_lag1`.


In [6]:
PREDICTOR_COLS = [
    # lag
    "poverty_lag1",
    # Crime
    "crime_rate_per_100k",
    "crime_selected_rate_per_100k",
    "crime_total",
    # Health capacity
    "hospitals_per_100k",
    "beds_per_10k",
    "hospitals",
    "beds",
    "Number of physicians",
    "Number of hospitalized patients",
    "Annual average occupancy of a bed",
    # Context/time
    "population",
    "year",
]

TARGET_COL = "poverty_rate"

# Keep only rows usable for modeling
df_model = df.dropna(subset=PREDICTOR_COLS + [TARGET_COL]).copy()

train_mask = df_model["year"] <= 2020
test_mask = df_model["year"] >= 2021

X_train = df_model.loc[train_mask, PREDICTOR_COLS].to_numpy(dtype=np.float64)
y_train = df_model.loc[train_mask, TARGET_COL].to_numpy(dtype=np.float64)
X_test = df_model.loc[test_mask, PREDICTOR_COLS].to_numpy(dtype=np.float64)
y_test = df_model.loc[test_mask, TARGET_COL].to_numpy(dtype=np.float64)

baseline_pred = df_model.loc[test_mask, "poverty_lag1"].to_numpy(dtype=np.float64)

baseline = {
    "r2": r2_score(y_test, baseline_pred),
    "mse": mean_squared_error(y_test, baseline_pred),
    "mae": mean_absolute_error(y_test, baseline_pred),
}
baseline


{'r2': 0.7278571711837595, 'mse': 45.93818181818182, 'mae': 5.10909090909091}

## Step 3 — Train a small MLP (PyTorch) with early stopping

We standardize features using **train-only** mean/std.

For early stopping, we hold out the **last train year** as a validation set.


In [7]:
import math
import random
from dataclasses import dataclass
from typing import List, Tuple

from torch import nn


SEED = 42


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def standardize_train_only(X_train: np.ndarray, X_val: np.ndarray, X_test: np.ndarray):
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)
    std = np.where(std == 0, 1.0, std)
    return (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std


def make_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "leaky_relu":
        return nn.LeakyReLU(negative_slope=0.01)
    if name == "elu":
        return nn.ELU()
    if name == "sigmoid":
        return nn.Sigmoid()
    # “mahout” isn’t a standard activation; include modern smooth alternatives
    if name == "mish":
        return nn.Mish()
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


class MLPRegressor(nn.Module):
    def __init__(self, in_dim: int, hidden_dims: Tuple[int, ...], activation: nn.Module, dropout: float = 0.0):
        super().__init__()
        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(activation)
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


@dataclass(frozen=True)
class TrainConfig:
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 32
    max_epochs: int = 2000
    patience: int = 100
    min_delta: float = 1e-5


def to_tensor(x: np.ndarray, device: torch.device) -> torch.Tensor:
    return torch.tensor(x, dtype=torch.float32, device=device)


def iter_minibatches(X: torch.Tensor, y: torch.Tensor, batch_size: int, rng: np.random.Generator):
    n = X.shape[0]
    idx = np.arange(n)
    rng.shuffle(idx)
    for start in range(0, n, batch_size):
        sl = idx[start : start + batch_size]
        yield X[sl], y[sl]


@torch.no_grad()
def eval_loss(model: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    pred = model(X)
    loss = nn.functional.mse_loss(pred, y)
    return float(loss.detach().cpu().item())


def train_mlp(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor, X_val: torch.Tensor, y_val: torch.Tensor, cfg: TrainConfig, seed: int):
    rng = np.random.default_rng(seed)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_state = None
    best_val = math.inf
    no_improve = 0

    for _epoch in range(cfg.max_epochs):
        model.train()
        for xb, yb in iter_minibatches(X_train, y_train, cfg.batch_size, rng):
            optim.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = nn.functional.mse_loss(pred, yb)
            loss.backward()
            optim.step()

        val_loss = eval_loss(model, X_val, y_val)
        if val_loss < best_val - cfg.min_delta:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= cfg.patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict(model: nn.Module, X: torch.Tensor) -> np.ndarray:
    model.eval()
    return model(X).detach().cpu().numpy()


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mse": float(mean_squared_error(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


## Step 4 — Activation/architecture sweep

We evaluate a grid of architectures and activations:

- activations: `relu`, `tanh`, `leaky_relu`, `elu`, `sigmoid`, `mish`, `gelu`
- hidden sizes: `(16,)`, `(32,)`, `(64,)`, `(32,16)`, `(64,32)`

Then we compare everything to the `poverty_lag1` baseline.


In [8]:
set_seed(SEED)

# Validation year = last year in train (to keep time ordering)
val_year = int(df_model.loc[train_mask, "year"].max())
train2_mask = df_model["year"] < val_year
val_mask = df_model["year"] == val_year

X_train2 = df_model.loc[train2_mask, PREDICTOR_COLS].to_numpy(dtype=np.float64)
y_train2 = df_model.loc[train2_mask, TARGET_COL].to_numpy(dtype=np.float64)
X_val = df_model.loc[val_mask, PREDICTOR_COLS].to_numpy(dtype=np.float64)
y_val = df_model.loc[val_mask, TARGET_COL].to_numpy(dtype=np.float64)

X_train2_s, X_val_s, X_test_s = standardize_train_only(X_train2, X_val, X_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_t = to_tensor(X_train2_s, device)
y_train_t = to_tensor(y_train2, device)
X_val_t = to_tensor(X_val_s, device)
y_val_t = to_tensor(y_val, device)
X_test_t = to_tensor(X_test_s, device)

cfg = TrainConfig()
activations = ["relu", "tanh", "leaky_relu", "elu", "sigmoid", "mish", "gelu"]
hidden_grid: List[Tuple[int, ...]] = [(16,), (32,), (64,), (32, 16), (64, 32)]

rows = [
    {
        "model": "baseline",
        "activation": "na",
        "hidden_dims": "na",
        "val_year": val_year,
        **compute_metrics(y_test, baseline_pred),
    }
]

in_dim = X_train_t.shape[1]
for act_name in activations:
    act = make_activation(act_name)
    for hidden_dims in hidden_grid:
        model = MLPRegressor(in_dim=in_dim, hidden_dims=hidden_dims, activation=act).to(device)
        model = train_mlp(model, X_train_t, y_train_t, X_val_t, y_val_t, cfg, seed=SEED)
        test_pred = predict(model, X_test_t)
        rows.append(
            {
                "model": "mlp",
                "activation": act_name,
                "hidden_dims": "x".join(map(str, hidden_dims)),
                "val_year": val_year,
                **compute_metrics(y_test, test_pred),
            }
        )

results = pd.DataFrame(rows).sort_values(["r2", "mse"], ascending=[False, True]).reset_index(drop=True)
results.head(15)


,model,activation,hidden_dims,val_year,r2,mse,mae
0,baseline,na,na,2020,0.727857,45.938182,5.109091
1,mlp,elu,64x32,2020,0.542928,77.154564,6.699696
2,mlp,elu,32,2020,0.503855,83.750096,7.118019
3,mlp,tanh,16,2020,0.501395,84.165338,7.109382
4,mlp,elu,16,2020,0.500973,84.236706,7.100941
5,mlp,tanh,64,2020,0.490985,85.922675,7.357670
6,mlp,elu,64,2020,0.480713,87.656556,7.415634
7,mlp,sigmoid,32,2020,0.478309,88.062329,7.072110
8,mlp,gelu,16,2020,0.462767,90.685815,7.175257
9,mlp,tanh,32,2020,0.461991,90.816781,7.869850


In [9]:
# Save results in the same place as the script
OUT_PATH = "../data/processed/results/poverty_nn_activation_sweep.csv"
results.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)


Saved: ../data/processed/results/poverty_nn_activation_sweep.csv


## What we got (and why)

### Summary of results
- **Baseline (`poverty_lag1`)** performed best on the held-out test years.
- The tested **PyTorch MLPs (different activations/hidden sizes)** did **not** beat the baseline on this dataset/time split.

### Why a neural network didn’t improve R² here
This is a common outcome in time-series / panel forecasting when:

- **The baseline is extremely strong**: poverty is highly persistent year-to-year, so `poverty_lag1` already captures most predictable signal.
- **The dataset is small**: after lagging + time split, the training set is limited (marzes × years). Neural nets generally need *much more* data to generalize.
- **Extra predictors add limited new information** beyond last year’s poverty at this aggregation level, so the NN tends to learn noise and underperform.

### What would actually improve performance (next steps)
To beat `poverty_lag1`, we need **new predictive signal** and/or **more data**, not just a more complex model:

- **Better predictors** (available at forecast time): GDP growth, inflation, unemployment, remittances, exchange rate, policy/aid variables, shock indicators, etc.
- **More observations**: more years, more granular geography, or higher-frequency data.
- **Better targets/features**:
  - predict the change: \(\Delta_t = poverty_t - poverty_{t-1}\) and then \(\hat{poverty}_t = poverty_{t-1} + \hat{\Delta}_t\)
  - add more lags (`poverty_lag2`, `poverty_lag3`) and rolling features (trend/mean)

### Important caution: avoid leakage
Any feature that is mathematically tied to `poverty_rate` in the same year (or computed from it) can inflate R² unrealistically. Always ensure features reflect information available **before** the year being predicted.
